In [0]:
path = "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/"
display(dbutils.fs.ls(path))

path,name,size,modificationTime
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/customers/,customers/,0,1790835640000
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/,orders/,0,1790835647000
abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/products/,products/,0,1790835655000


In [0]:
orders_path = "abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/landing/orders/year=2026/month=10/day=01/"

df = (
    spark.read
    .option("multiLine", True)
    .json(orders_path)
)

df.printSchema()

print("Record count:", df.count())

display(df)

df.write.format("delta").mode("overwrite").save("abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/orders/")

root
 |-- customer_id: long (nullable = true)
 |-- market: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- sales_amount: long (nullable = true)
 |-- updated_at: string (nullable = true)

Record count: 5


customer_id,market,order_id,order_timestamp,product_id,quantity,sales_amount,updated_at
101,India,5001,2026-10-01T07:55:00,501,2,5000,2026-10-01T07:57:00
102,India,5002,2026-10-01T07:58:00,502,3,3000,2026-10-01T07:59:00
103,USA,5003,2026-10-01T08:02:00,503,1,7000,2026-10-01T08:03:00
104,UK,5004,2026-10-01T08:05:00,504,4,2500,2026-10-01T08:06:00
101,India,5005,2026-10-01T08:10:00,505,2,4500,2026-10-01T08:11:00


In [0]:
df_bronze=spark.read.format("delta").load("abfss://customer-sales-datalake@nestlesalesdatalake.dfs.core.windows.net/bronze/orders/")
df_bronze.printSchema()
df_bronze.display()

root
 |-- customer_id: long (nullable = true)
 |-- market: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- sales_amount: long (nullable = true)
 |-- updated_at: string (nullable = true)



customer_id,market,order_id,order_timestamp,product_id,quantity,sales_amount,updated_at
101,India,5001,2026-10-01T07:55:00,501,2,5000,2026-10-01T07:57:00
102,India,5002,2026-10-01T07:58:00,502,3,3000,2026-10-01T07:59:00
103,USA,5003,2026-10-01T08:02:00,503,1,7000,2026-10-01T08:03:00
104,UK,5004,2026-10-01T08:05:00,504,4,2500,2026-10-01T08:06:00
101,India,5005,2026-10-01T08:10:00,505,2,4500,2026-10-01T08:11:00


In [0]:
from pyspark.sql.functions import to_timestamp,col

In [0]:
df_bronze_transformed = (
    df_bronze.withColumn(
    "order_timestamp",
    to_timestamp(col("order_timestamp"), "yyyy-MM-dd'T'HH:mm:ss"))
    .withColumn("updated_at",to_timestamp(col("updated_at"), "yyyy-MM-dd'T'HH:mm:ss"))
)
df_bronze_transformed.printSchema()
df_bronze_transformed.display()    
df_invalid = df_bronze_transformed.filter(
    col("order_id").isNull()
    | col("customer_id").isNull()
    | (col("quantity") <= 0)
    | (col("sales_amount") <= 0)
)
display(df_invalid)                                                             

root
 |-- customer_id: long (nullable = true)
 |-- market: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- sales_amount: long (nullable = true)
 |-- updated_at: timestamp (nullable = true)



customer_id,market,order_id,order_timestamp,product_id,quantity,sales_amount,updated_at
101,India,5001,2026-10-01T07:55:00.000Z,501,2,5000,2026-10-01T07:57:00.000Z
102,India,5002,2026-10-01T07:58:00.000Z,502,3,3000,2026-10-01T07:59:00.000Z
103,USA,5003,2026-10-01T08:02:00.000Z,503,1,7000,2026-10-01T08:03:00.000Z
104,UK,5004,2026-10-01T08:05:00.000Z,504,4,2500,2026-10-01T08:06:00.000Z
101,India,5005,2026-10-01T08:10:00.000Z,505,2,4500,2026-10-01T08:11:00.000Z


customer_id,market,order_id,order_timestamp,product_id,quantity,sales_amount,updated_at
